# LangChain

This notebook provides hands-on practice to become familiar with **LangChain** and the core concepts used to build LLM-powered applications.

## What is LangChain?

**LangChain** is a framework for building applications powered by large language models (LLMs).

It provides structured components for working with prompts, models, tools, memory, and external data sources, making it easier to create real-world LLM use cases such as chatbots, agents, and retrieval-augmented generation (RAG) systems.

## Installations & Imports

In [1]:
!pip -q install langchain langchain_openai langchain-community huggingface_hub openai

In [ ]:
!pip show langchain

Name: langchain
Version: 1.2.13
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /opt/miniconda3/envs/mlpeople5/lib/python3.11/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [1]:
%load_ext autoreload
%autoreload 2

# LLMs
from langchain_openai import ChatOpenAI

# Prompts
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder

# Messages
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# Output parsers (very important for LCEL)
from langchain_core.output_parsers import StrOutputParser

# Runnables (LCEL)
from langchain_core.runnables import RunnablePassthrough

overall_temperature = 0.1

## Create Model

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=overall_temperature,
)

In [13]:
llm

ChatOpenAI(profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x1251cb210>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1251cb190>, root_client=<openai.OpenAI object at 0x1251c9a10>, root_async_client=<openai.AsyncOpenAI object at 0x1251ca310>, model_name='gpt-4o-mini', temperature=0.1, model_kwargs={'max_output_tokens': 400}, openai_a

### Get Response

llm.invoke returns AIMessage type

In [15]:
llm.invoke?

Signature:
llm.invoke(
    input: 'LanguageModelInput',
    config: 'RunnableConfig | None' = None,
    *,
    stop: 'list[str] | None' = None,
    **kwargs: 'Any',
) -> 'AIMessage'
Docstring:
Transform a single input into an output.

Args:
    input: The input to the `Runnable`.
    config: A config to use when invoking the `Runnable`.

        The config supports standard keys like `'tags'`, `'metadata'` for
        tracing purposes, `'max_concurrency'` for controlling how much work to
        do in parallel, and other keys.

        Please refer to `RunnableConfig` for more details.

Returns:
    The output of the `Runnable`.
File:      /opt/miniconda3/envs/mlpeople5/lib/python3.11/site-packages/langchain_core/language_models/chat_models.py
Type:      method

In [10]:
llm.invoke("Explain what LangChain is in one sentence")

AIMessage(content='LangChain is a framework designed to facilitate the development of applications that leverage large language models (LLMs) by providing tools for chaining together various components such as prompts, memory, and APIs.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 15, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPTmOjiEi72shzHwNxYSiZ1IfovW6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d4429-30d4-7f90-b160-d66f8ec62e74-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 38, 'total_tokens': 53, 'input_token_details': 

In [17]:
pasta_question_input = "What are 5 vacation destinations for someone who likes to eat pasta?"
pasta_question_output = llm.invoke(pasta_question_input)
pasta_question_output

AIMessage(content="If you love pasta, there are several fantastic vacation destinations where you can indulge in delicious pasta dishes. Here are five top choices:\n\n1. **Italy**: The birthplace of pasta, Italy is a must-visit for any pasta lover. Regions like Emilia-Romagna are famous for their handmade pasta, such as tagliatelle and tortellini. Cities like Bologna, Florence, and Naples offer a wide variety of pasta dishes, from classic spaghetti carbonara to regional specialties like pici and orecchiette.\n\n2. **San Francisco, California**: Known for its vibrant food scene, San Francisco boasts a variety of Italian restaurants that serve authentic pasta dishes. The North Beach neighborhood is particularly famous for its Italian cuisine, where you can enjoy everything from fresh linguine to rich lasagna.\n\n3. **Buenos Aires, Argentina**: Argentina has a strong Italian influence, and you can find excellent pasta dishes throughout Buenos Aires. Try the local version of pasta, such as

In [18]:
print(pasta_question_output.content)

If you love pasta, there are several fantastic vacation destinations where you can indulge in delicious pasta dishes. Here are five top choices:

1. **Italy**: The birthplace of pasta, Italy is a must-visit for any pasta lover. Regions like Emilia-Romagna are famous for their handmade pasta, such as tagliatelle and tortellini. Cities like Bologna, Florence, and Naples offer a wide variety of pasta dishes, from classic spaghetti carbonara to regional specialties like pici and orecchiette.

2. **San Francisco, California**: Known for its vibrant food scene, San Francisco boasts a variety of Italian restaurants that serve authentic pasta dishes. The North Beach neighborhood is particularly famous for its Italian cuisine, where you can enjoy everything from fresh linguine to rich lasagna.

3. **Buenos Aires, Argentina**: Argentina has a strong Italian influence, and you can find excellent pasta dishes throughout Buenos Aires. Try the local version of pasta, such as ñoquis (gnocchi) and sor

In [ ]:
def query_llm(question: str) -> AIMessage:
    """
    Sends a question to the ChatOpenAI model, prints the response text, and returns the full AIMessage.

    Args:
        question (str): The text question or prompt to send to the model.

    Returns:
        AIMessage: The model's response object containing content, role, and other metadata.
    """
    # Invoke the model
    response: AIMessage = llm.invoke(question)

    # Print the actual text
    print(response.content)

    return response

In [20]:
query_llm("who am I talking to?")

You’re talking to an AI language model created by OpenAI. I'm here to help answer your questions and provide information on a wide range of topics. How can I assist you today?


AIMessage(content="You’re talking to an AI language model created by OpenAI. I'm here to help answer your questions and provide information on a wide range of topics. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 13, 'total_tokens': 50, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPTzFYOy3VtDGzBnjYSFTsWdASk5x', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d4435-5cf0-7ca0-b9a1-fdca512885d1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 37, 'total_tokens': 50, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_tok

### Stream

In [21]:
llm.stream?

Signature:
llm.stream(
    input: 'LanguageModelInput',
    config: 'RunnableConfig | None' = None,
    *,
    stop: 'list[str] | None' = None,
    **kwargs: 'Any',
) -> 'Iterator[AIMessageChunk]'
Docstring:
Default implementation of `stream`, which calls `invoke`.

Subclasses must override this method if they support streaming output.

Args:
    input: The input to the `Runnable`.
    config: The config to use for the `Runnable`.
    **kwargs: Additional keyword arguments to pass to the `Runnable`.

Yields:
    The output of the `Runnable`.
File:      /opt/miniconda3/envs/mlpeople5/lib/python3.11/site-packages/langchain_core/language_models/chat_models.py
Type:      method

In [ ]:
question = "Give me a 5-line poem about pasta."

# Stream the response token by token
for event in llm.stream(question):
    # Each event is a partial update from the model
    if hasattr(event, "content") and event.content:
        print(event.content, end="", flush=True)

print()  # newline at the end

In boiling water, dreams entwine,  
Golden strands in a dance divine.  
With sauce that whispers, herbs that sing,  
Each twirl a joy, each bite a fling.  
Pasta, a love that time can't confine.


In [26]:
for event in llm.stream(
    "What are some theories about the relationship between unemployment and inflation?"
):
    if hasattr(event, "content") and event.content:
        print(event.content, end="", flush=True)

print()  # newline at the end

The relationship between unemployment and inflation has been a central topic in economics, particularly through the lens of the Phillips Curve and other theories. Here are some key theories and concepts that explain this relationship:

1. **Phillips Curve**: Originally proposed by A.W. Phillips in 1958, the Phillips Curve suggests an inverse relationship between unemployment and inflation. According to this theory, when unemployment is low, inflation tends to be high, and vice versa. The rationale is that low unemployment leads to increased demand for labor, which can drive up wages and, subsequently, prices.

2. **Expectations-Augmented Phillips Curve**: This version of the Phillips Curve incorporates inflation expectations. It suggests that if people expect higher inflation in the future, they will demand higher wages, which can lead to a self-fulfilling cycle of rising inflation. This theory implies that the trade-off between unemployment and inflation may only hold in the short run

## Prompts: Managing requests for LLMs

In [ ]:
prompt = PromptTemplate(
    input_variables=["food"],            # variables you will fill later
    template="What are 5 vacation destinations for someone who likes to eat {food}?",
)

print(prompt.format(food="donuts"))

What are 5 vacation destinations for someone who likes to eat donuts?


In [42]:
question_prompt = prompt.format(food="donuts")
response = llm.invoke(question_prompt)
print(response.content)

If you love donuts, there are several vacation destinations known for their delicious and unique donut offerings. Here are five great places to consider:

1. **Portland, Oregon**: Known for its vibrant food scene, Portland is home to several famous donut shops, including Voodoo Doughnut and Blue Star Donuts. You can find a wide variety of creative flavors and toppings, making it a donut lover's paradise.

2. **Los Angeles, California**: LA boasts a diverse array of donut shops, from classic establishments like Randy's Donuts to trendy spots like The Donut Man, famous for its fresh strawberry-filled donuts. The city's eclectic food culture means you'll find both traditional and innovative donut options.

3. **New York City, New York**: NYC is filled with iconic donut shops, such as Dough and The Doughnut Project. You can explore various neighborhoods to discover unique flavors and styles, from classic glazed to gourmet creations.

4. **Chicago, Illinois**: Chicago has a rich donut cultu

## Chains

In [43]:
# Prepare chain
chain = RunnablePassthrough(lambda food: prompt.format(food=food)) | llm

# Invoke chain
response = chain.invoke("fresh fish")
print(response.content)

Fresh fish refers to fish that has been recently caught and has not undergone any preservation processes like freezing or smoking. It is often considered to have superior flavor and texture compared to frozen or processed fish. When purchasing fresh fish, it's important to look for certain indicators of freshness, such as:

1. **Smell**: Fresh fish should have a mild, ocean-like smell. A strong, fishy odor is a sign that the fish is not fresh.

2. **Eyes**: The eyes of fresh fish should be clear and bulging, not cloudy or sunken.

3. **Gills**: The gills should be bright red or pink, indicating freshness. Brown or dull gills suggest that the fish is old.

4. **Flesh**: The flesh should be firm and bounce back when pressed. It should not be slimy or discolored.

5. **Scales**: The scales should be shiny and intact, not dull or falling off.

Fresh fish can be prepared in various ways, including grilling, baking, frying, or steaming, and is often used in a wide range of cuisines around th

In [45]:
chain = prompt | llm
print(chain.invoke("burgers").content)

If you're a burger enthusiast, there are several fantastic vacation destinations known for their delicious burger offerings. Here are five great places to consider:

1. **Los Angeles, California**: Known for its diverse food scene, LA boasts a plethora of gourmet burger joints. From classic diners to upscale burger restaurants, you can find everything from traditional cheeseburgers to innovative creations. Don't miss places like In-N-Out Burger, Umami Burger, and Father's Office.

2. **Chicago, Illinois**: Chicago is famous for its deep-dish pizza, but it also has a vibrant burger culture. You can explore iconic spots like Au Cheval, known for its decadent burgers, and Kuma's Corner, which offers heavy metal-themed burgers. The city also hosts various burger festivals throughout the year.

3. **Austin, Texas**: Austin is a haven for food lovers, and its burger scene is no exception. With a mix of food trucks and sit-down restaurants, you can enjoy unique takes on the classic burger. Ch

## Agents

**Agents** are LLM-powered controllers that can **decide which tools to use**, **in what order**, and **with what inputs** to answer a user’s question.

Unlike simple chains (prompt → model → output), an agent can:

* plan steps,
* call external tools (calculator, Python, web search, APIs),
* observe tool results,
* iterate if needed,
* produce a final answer.

In LangChain v1, agents are built on top of **LangGraph**, where:

* the **model** decides what to do next,
* **tools** execute actions,
* the agent loops until the task is complete.

This enables **reasoning + acting** (ReAct pattern): the model reasons about the problem, acts via tools, observes results, and continues reasoning.


In [2]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=overall_temperature)

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))

@tool
def python_interpreter(code: str) -> str:
    """Execute Python code. Use print() to output results."""
    local_vars = {}
    exec(code, {}, local_vars)
    return str(local_vars)

agent = create_agent(
    model=llm,
    tools=[calculator, python_interpreter],
    system_prompt="You are a helpful assistant that can use tools for math and Python code"
)

In [4]:
invoke_result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is 25 * 17?"}]}
)
invoke_result

{'messages': [HumanMessage(content='What is 25 * 17?', additional_kwargs={}, response_metadata={}, id='1c57a3af-b02e-4b61-9561-a133713f1a82'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 90, 'total_tokens': 107, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPql4CyCLWPgGvzA1kILUUUC95lEs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d496c-fff1-7952-bf04-dfe9491c86c0-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '25 * 17'}, 'id': 'call_BUa4OXyW2gl8SauQIWiBkrP0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_toke

In [6]:
for key in invoke_result.keys():
    print(key)

messages


In [5]:
invoke_result["messages"]

[HumanMessage(content='What is 25 * 17?', additional_kwargs={}, response_metadata={}, id='1c57a3af-b02e-4b61-9561-a133713f1a82'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 90, 'total_tokens': 107, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPql4CyCLWPgGvzA1kILUUUC95lEs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d496c-fff1-7952-bf04-dfe9491c86c0-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '25 * 17'}, 'id': 'call_BUa4OXyW2gl8SauQIWiBkrP0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_tokens': 17, 'tota

In [7]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is 25 * 17?"}]},
    stream_mode="updates"
):
    print(chunk, end="\n")

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 90, 'total_tokens': 107, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_cd4a20171f', 'id': 'chatcmpl-DPqnJ7LzTolmGniHg7b36hAOaCvWI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d496f-1d9e-7ef2-8f25-81f33ce78707-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '25 * 17'}, 'id': 'call_7wJWXPNSq1mdIh9xmKg5oxzM', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_tokens': 17, 'total_tokens': 107, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is 25 * 17?"}]},
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'name': 'calculator', 'args': {'expression': '25 * 17'}, 'id': 'call_8LGfZfEolvPrNBnHTk17vh0K'}]
step: tools
content: [{'type': 'text', 'text': '425'}]
step: model
content: [{'type': 'text', 'text': 'The result of \\( 25 \\times 17 \\) is 425.'}]


### Define tools

In [88]:
!pip install -q google-search-results

In [9]:
from serpapi import GoogleSearch
import os

@tool
def calculator(expression: str) -> str:
    """Evaluate a simple math expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"
    

@tool
def python_interpreter(code: str) -> str:
    """Execute Python code using print() to return results."""
    local_vars = {}
    try:
        exec(code, {}, local_vars)
        return str(local_vars)
    except Exception as e:
        return f"Error: {e}"


@tool
def web_search(query: str) -> str:
    """Search the web using SerpAPI."""
    params = {
        "q": query,
        "api_key": os.getenv("SERPAPI_API_KEY"),
        "engine": "google",
        "num": 3,
    }
    search = GoogleSearch(params)
    results = search.get_dict()
    snippets = [r.get("snippet", "") for r in results.get("organic_results", [])[:3]]
    return "\n".join(snippets)

### Create LLM and agent

In [10]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=overall_temperature, streaming=True)

tools = [calculator, python_interpreter, web_search]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that can use tools for math, Python code, and web search."
)

In [14]:
user_question_1 = "Calculate sqrt(144) + 20 using the calculator tool. Also search the latest SpaceX news."

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": user_question_1}]},
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'name': 'calculator', 'args': {'expression': 'sqrt(144) + 20'}, 'id': 'call_ByRUDBDTfiHJosXcKjffKUom'}, {'type': 'tool_call', 'name': 'web_search', 'args': {'query': 'latest SpaceX news'}, 'id': 'call_oTrFzhdGMhHeom817ORQ8nIX'}]
step: tools
content: [{'type': 'text', 'text': "Error: name 'sqrt' is not defined"}]
step: tools
content: [{'type': 'text', 'text': 'SpaceX has acquired xAI to form the most ambitious, vertically-integrated innovation engine on (and off) Earth, with AI, rockets, space-based internet, ...'}]
step: model
content: [{'type': 'tool_call', 'name': 'calculator', 'args': {'expression': '(144)**0.5 + 20'}, 'id': 'call_AfAfNZzK4hVVfQl6v2B0LROq'}, {'type': 'tool_call', 'name': 'web_search', 'args': {'query': 'latest SpaceX news'}, 'id': 'call_m0ZF6Gkt6QwKtlrOZxpSgcPo'}]
step: tools
content: [{'type': 'text', 'text': '32.0'}]
step: tools
content: [{'type': 'text', 'text': 'SpaceX has acquired xAI to form the most ambitious, verti

In [15]:
agent_invoke_result1 = agent.invoke({"messages": [{"role": "user", "content": user_question_1}]})


In [17]:
print(agent_invoke_result1["messages"][-1].content)

The result of the calculation \( \sqrt{144} + 20 \) is \( 32.0 \).

In the latest SpaceX news, the company has acquired xAI to form a highly ambitious, vertically-integrated innovation engine focused on both Earth and space, incorporating AI, rockets, and space-based internet.


### “verbose” demo

In [20]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def pretty_print_agent_result(result):
    print("\n=== AGENT TRACE ===\n")

    for msg in result["messages"]:

        if isinstance(msg, HumanMessage):
            print(f"[USER]\n{msg.content}\n")

        elif isinstance(msg, AIMessage):
            # Tool call request (ReAct thinking step)
            if msg.tool_calls:
                for call in msg.tool_calls:
                    print(f"[TOOL CALL] {call['name']}({call['args']})\n")

            # Final LLM response
            elif msg.content.strip():
                print(f"[LLM]\n{msg.content}\n")

        elif isinstance(msg, ToolMessage):
            print(f"[TOOL RESULT] ({msg.name})\n{msg.content}\n")

    print("=== END TRACE ===\n")

In [21]:
pretty_print_agent_result(agent_invoke_result1)


=== AGENT TRACE ===

[USER]
Calculate sqrt(144) + 20 using the calculator tool. Also search the latest SpaceX news.

[TOOL CALL] calculator({'expression': 'sqrt(144) + 20'})

[TOOL CALL] web_search({'query': 'latest SpaceX news'})

[TOOL RESULT] (calculator)
Error: name 'sqrt' is not defined

[TOOL RESULT] (web_search)
SpaceX has acquired xAI to form the most ambitious, vertically-integrated innovation engine on (and off) Earth, with AI, rockets, space-based internet, ...

[TOOL CALL] calculator({'expression': '(144)**0.5 + 20'})

[TOOL CALL] web_search({'query': 'latest SpaceX news'})

[TOOL RESULT] (calculator)
32.0

[TOOL RESULT] (web_search)
SpaceX has acquired xAI to form the most ambitious, vertically-integrated innovation engine on (and off) Earth, with AI, rockets, space-based internet, ...

[LLM]
The result of the calculation \( \sqrt{144} + 20 \) is \( 32.0 \).

In the latest SpaceX news, the company has acquired xAI to form a highly ambitious, vertically-integrated innovati

### Streaming “verbose” demo

In [32]:
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage


def pretty_print_agent_stream(agent, user_question):
    print("\n=== STREAMING AGENT TRACE ===\n")

    for chunk in agent.stream(
        {"messages": [{"role": "user", "content": user_question}]}
    ):
        # Each chunk is: {'model': {...}} OR {'tools': {...}}
        for node_name, node_data in chunk.items():
            messages = node_data.get("messages", [])

            for msg in messages:
                # USER (rarely appears in stream)
                if isinstance(msg, HumanMessage):
                    print(f"[USER]\n{msg.content}\n")

                # LLM output or tool calls
                elif isinstance(msg, AIMessage):
                    if msg.tool_calls:
                        for call in msg.tool_calls:
                            print(f"[TOOL CALL] {call['name']}({call['args']})\n")
                    elif msg.content.strip():
                        print(f"[LLM]\n{msg.content}\n")

                # Tool result
                elif isinstance(msg, ToolMessage):
                    print(f"[TOOL RESULT] ({msg.name})\n{msg.content}\n")

    print("=== END STREAM ===\n")

In [ ]:
user_question_2 = "Who is the current leader of China? What is the largest prime number that is smaller than their age?"

pretty_print_agent_stream(agent, user_question_2)


=== STREAMING AGENT TRACE ===

[TOOL CALL] web_search({'query': 'current leader of China 2023'})

[TOOL CALL] python_interpreter({'code': "from sympy import prevprime\nage = 70  # Xi Jinping's age as of 2023\nlargest_prime = prevprime(age)\nlargest_prime"})

[TOOL RESULT] (python_interpreter)
{'prevprime': <function prevprime at 0x1277e8900>, 'age': 70, 'largest_prime': 67}

[TOOL RESULT] (web_search)
Xi Jinping (born 15 June 1953) is a Chinese statesman and politician who has served as the general secretary of the Chinese Communist Party (CCP) and ...
Xi Jinping is a politician and government official who became president of China in 2013 and general secretary of the Chinese Communist Party in ...
1983–present) ; Xi Jinping 习近平 (born 1953) · 17 March 2018, 10 March 2023 ; Xi Jinping 习近平 (born 1953) · 10 March 2023, Incumbent ...

[LLM]
The current leader of China is Xi Jinping, who was born on June 15, 1953. As of 2023, he is 70 years old.

The largest prime number that is smaller th

In [34]:
user_question_3 = "Find the population of Canada and calculate 3% of it. Then find the area of the circle with the radius value equal to found number"

pretty_print_agent_stream(agent, user_question_3)


=== STREAMING AGENT TRACE ===

[TOOL CALL] web_search({'query': 'current population of Canada 2023'})

[TOOL CALL] calculator({'expression': '3% of x'})

[TOOL RESULT] (calculator)
Error: invalid syntax (<string>, line 1)

[TOOL RESULT] (web_search)
Population of Canada (2026 and historical) ; 2023, 39,299,105, 1.23% ; 2022, 38,821,259, 0.85% ; 2020, 38,171,902, 1.2% ; 2015, 35,962,234, 1.01% ...
Canada's population was estimated at 40,528,396 on October 1, 2023, an increase of 430,635 people (+1.1%) from July 1. This was the highest ...
Total population for Canada in 2023 was 40,097,761, a 2.98% increase from 2022. Total population for Canada in 2022 was 38,939,056, a 1.83% increase from 2021.

[TOOL CALL] calculator({'expression': '3% of 40528396'})

[TOOL RESULT] (calculator)
Error: invalid syntax (<string>, line 1)

[TOOL CALL] calculator({'expression': '0.03 * 40528396'})

[TOOL CALL] calculator({'expression': 'pi * (40528396 ** 2)'})

[TOOL RESULT] (calculator)
1215851.88

[TOOL

In [35]:
user_question_4 = "Find today’s weather in Tokyo and compute the difference between the temperature in Celsius and Fahrenheit."

pretty_print_agent_stream(agent, user_question_4)


=== STREAMING AGENT TRACE ===

[TOOL CALL] web_search({'query': 'Tokyo weather today'})

[TOOL CALL] calculator({'expression': '(F - 32) * 5/9 = C'})

[TOOL RESULT] (calculator)
Error: invalid syntax (<string>, line 1)

[TOOL RESULT] (web_search)
Hourly Weather · 1 AM 51°. rain drop 73% · 2 AM 51°. rain drop 73% · 3 AM 50°. rain drop 49% · 4 AM 50°. rain drop 49% · 5 AM 50°. rain drop 49% · 6 AM 50°.
Cloudy skies. Low 59F. Winds NW at 5 to 10 mph. Humidity. 82%. UV Index. 0 of 11. Moonrise. 4:17 pm. Moonset. 4: ...

[TOOL CALL] web_search({'query': 'Tokyo weather today temperature in Celsius and Fahrenheit'})

[TOOL RESULT] (web_search)
... becoming windy in the afternoon with times of clouds and sun Hi: 58° · Current Weather. 7:41 PM. 53°F. Rain. RealFeel® 46° · MINUTECAST®. Rain for at least 60 ...
... (GMT +9) | Updated 55 seconds ago. --° | 50°. 52 °F. like 52°. icon. Rain. N. 3. Gusts 4 °mph. Tomorrow's temperature is forecast to be COOLER than today.
Weather in Tokyo, Japan ... 

## Memory

The simplest form of memory is simply passing chat history messages along a thread. Here's an example:

In [36]:
llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content="You are a helpful assistant. Answer all questions to the best of your ability."
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | llm

ai_msg = chain.invoke(
    {
        "messages": [
            HumanMessage(
                content="Translate from English to French: I love programming."
            ),
            AIMessage(content="J'adore la programmation."),
            HumanMessage(content="What did you just say?"),
        ],
    }
)
print(ai_msg)

content='I translated "I love programming" into French, which is "J\'adore la programmation."' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 56, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPrT7gqT6e0jeP0B05tmppAswEfbO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d4996-ae0a-7842-8e31-07bc79967e2a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 56, 'output_tokens': 18, 'total_tokens': 74, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [37]:
ai_msg.content

'I translated "I love programming" into French, which is "J\'adore la programmation."'

### Automatic history management

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langchain_core.messages import SystemMessage

# Create a graph workflow whose STATE is a list of chat messages.
# MessagesState is a predefined schema: {"messages": List[BaseMessage]}
workflow = StateGraph(state_schema=MessagesState)


# This function is a GRAPH NODE.
# It receives the current conversation STATE and must return
# an updated piece of state (here: new messages from the model).
def call_model(state: MessagesState):
    # System prompt is re-added on every model call.
    # It is NOT stored in memory — only user/AI messages are.
    system_prompt = (
        "You are a helpful assistant. "
        "Answer all questions to the best of your ability."
    )

    # The model must receive the full conversation history.
    # LangGraph keeps that history inside state["messages"].
    messages = [SystemMessage(content=system_prompt)] + state["messages"]

    # Call the LLM with the conversation so far
    response = llm.invoke(messages)

    # IMPORTANT:
    # A graph node must return a DICT that updates the state.
    # Here we append the model response into the "messages" state.
    return {"messages": response}


# Register the node in the graph under the name "model"
workflow.add_node("model", call_model)

# Define the graph flow:
# START → model
# (Whenever the app is invoked, it starts at START and goes to this node)
workflow.add_edge(START, "model")


# MemorySaver is a CHECKPOINTER.
# It automatically saves and restores the graph state between calls.
# This is the LangGraph way of implementing conversational memory.
memory = MemorySaver()

# Compile the workflow into a runnable app with memory enabled.
# From now on, app.invoke(...) will remember previous messages
# as long as the same thread_id is used.
app = workflow.compile(checkpointer=memory)

In [42]:
config = {"configurable": {"thread_id": "user_1"}}

app.invoke(
    {"messages": [HumanMessage(content="Hi, my name is Maksym. I was born before 2000")]},
    config=config,
)

app.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=config,
)

app.invoke(
    {"messages": [HumanMessage(content="And what is my age?")]},
    config=config,
)

{'messages': [HumanMessage(content='Hi, my name is Maksym. I was born before 2000', additional_kwargs={}, response_metadata={}, id='9484c8ee-43a1-4ff2-8cde-9501c52dcda7'),
  AIMessage(content="Hi Maksym! It's nice to meet you. Since you were born before 2000, you likely have some interesting experiences and perspectives shaped by growing up in the late 20th century and early 21st century. If you have any questions or topics you'd like to discuss, feel free to ask!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 42, 'total_tokens': 104, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPrdAdLeFz2GV0EWF5jwJaxnJSPos', 'service_tier': 'default', 'f


**Two types of memory**
| Type          | Where                 | How it works                         | Scope                   |
| ------------- | --------------------- | ------------------------------------ | ----------------------- |
| Prompt memory | `MessagesPlaceholder` | You manually pass messages           | One call only           |
| Graph memory  | `MemorySaver`         | LangGraph stores state between calls | Persistent conversation |


### Modifying Chat History
Modifying saved chat messages can help chatbot handle different situations.

#### Trimming Messages
LLMs and chat models have limited context windows, and even if you don’t exceed the limits, you can limit the number of distractions the model has to deal with. One solution is to trim the history messages before passing them to the model. Let’s look at the example of the history with the app we announced above:

In [44]:
demo_ephemeral_chat_history = [
    HumanMessage(content="Hey there! I'm Nemo."),
    AIMessage(content="Hello!"),
    HumanMessage(content="How are you today?"),
    AIMessage(content="Fine thanks!"),
]

app.invoke(
    {
        "messages": demo_ephemeral_chat_history
        + [HumanMessage(content="What's my name?")]
    },
    config={"configurable": {"thread_id": "user_2"}},
)

{'messages': [HumanMessage(content="Hey there! I'm Nemo.", additional_kwargs={}, response_metadata={}, id='553d30cd-da9f-4456-b688-706160110293'),
  AIMessage(content='Hello!', additional_kwargs={}, response_metadata={}, id='8fa48e1b-637a-4382-8885-95c35c0bbdf6', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='How are you today?', additional_kwargs={}, response_metadata={}, id='1ec48054-0932-4998-9ba5-be6d0bd95a69'),
  AIMessage(content='Fine thanks!', additional_kwargs={}, response_metadata={}, id='67a711e7-36a0-49f6-b6a6-af018d03948d', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}, id='1748f848-5d64-4380-924e-5b4f6afc8a73'),
  AIMessage(content='Your name is Nemo!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 63, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_

We can see that the program remembers the previously loaded name.

But let's imagine that we have a very small context window and we want to reduce the number of messages passed to the model to the last 2. We can use the built-in trim_messages utility to trim the messages based on their number of tokens before they reach our request. In this case, we will count each message as 1 "token" and keep only the last two messages:

In [45]:
from langchain_core.messages import trim_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langchain_core.messages import SystemMessage

# This component trims the conversation history before sending it to the model.
# - strategy="last": keeps the last messages
# - max_tokens=2: only keeps the last 2 messages
# - token_counter=len: counts each message as 1 token (simplified)
# This is useful to limit the context for large conversations.
trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

# Create a LangGraph workflow with the MessagesState schema
workflow = StateGraph(state_schema=MessagesState)

# Node function: called with the current conversation state
def call_model(state: MessagesState):
    # Apply trimming to reduce message history length
    trimmed_messages = trimmer.invoke(state["messages"])

    # System prompt added every call
    system_prompt = (
        "You are a helpful assistant. "
        "Answer all questions to the best of your ability."
    )

    # Concatenate system prompt with trimmed conversation messages
    messages = [SystemMessage(content=system_prompt)] + trimmed_messages

    # Call the LLM with the current context
    response = llm.invoke(messages)

    # Return updated state for the workflow
    return {"messages": response}

# Register node and edge in the graph
workflow.add_node("model", call_model)
workflow.add_edge(START, "model")

# Enable memory: saves conversation state between calls
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [47]:
app.invoke(
    {
        "messages": demo_ephemeral_chat_history
        + [HumanMessage(content="What is my name?")]
    },
    config={"configurable": {"thread_id": "user_3"}},
)

{'messages': [HumanMessage(content="Hey there! I'm Nemo.", additional_kwargs={}, response_metadata={}, id='553d30cd-da9f-4456-b688-706160110293'),
  AIMessage(content='Hello!', additional_kwargs={}, response_metadata={}, id='8fa48e1b-637a-4382-8885-95c35c0bbdf6', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='How are you today?', additional_kwargs={}, response_metadata={}, id='1ec48054-0932-4998-9ba5-be6d0bd95a69'),
  AIMessage(content='Fine thanks!', additional_kwargs={}, response_metadata={}, id='67a711e7-36a0-49f6-b6a6-af018d03948d', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='b1ed5687-8faa-4001-9e91-d06ee6f2affa'),
  AIMessage(content="I'm sorry, but I don't have access to personal information about you unless you share it with me. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 39, 'tot

#### Message History Summaries

We can use this prompt in other cases as well. For example, we can use an additional LLM call to generate a conversation summary before calling our application. Let's play back our chat history:

In [56]:
from langchain_core.messages import HumanMessage, RemoveMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

# Create the workflow graph with message-based state
workflow = StateGraph(state_schema=MessagesState)

# Node function: called with the current conversation state
def call_model(state: MessagesState):
    # System prompt for the assistant
    system_prompt = (
        "You are a helpful assistant. "
        "Answer all questions to the best of your ability. "
        "The provided chat history includes a summary of the earlier conversation."
    )
    system_message = SystemMessage(content=system_prompt)

    # Get the previous messages except the most recent one
    message_history = state["messages"][:-1]

    # If chat history gets long, summarize older messages
    if len(message_history) >= 4:
        last_human_message = state["messages"][-1]

        # Prompt to summarize past messages
        summary_prompt = (
            "Distill the above chat messages into a single summary message. "
            "Include as many specific details as you can."
        )

        # Call LLM to generate summary of older messages
        summary_message = llm.invoke(
            message_history + [HumanMessage(content=summary_prompt)]
        )

        # Remove older messages from memory (optional cleanup)
        delete_messages = [RemoveMessage(id=m.id) for m in state["messages"]]

        # Keep the latest human message
        human_message = HumanMessage(content=last_human_message.content)

        # Call LLM with system message, summary, and latest human message
        response = llm.invoke([system_message, summary_message, human_message])

        # Update messages: summary + latest human message + new response + cleanup
        message_updates = [summary_message, human_message, response] + delete_messages
    else:
        # If chat history is short, just process all messages normally
        message_updates = llm.invoke([system_message] + state["messages"])

    return {"messages": message_updates}

# Add node and edge to the graph
workflow.add_node("model", call_model)
workflow.add_edge(START, "model")

# Enable memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [57]:
app.invoke(
    {
        "messages": demo_ephemeral_chat_history
        + [HumanMessage("What did I say my name was?")]
    },
    config={"configurable": {"thread_id": "user_4"}},
)

{'messages': [AIMessage(content='Nemo greeted the assistant and asked how it was doing, to which the assistant responded that it was fine.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 60, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPrupkTFbG0NDtTEMaXbEaAqWmqdE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d49b0-e3f8-7612-b9c7-8da2dbddbba6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 22, 'total_tokens': 82, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
  HumanMess

### Memory Patterns in LangGraph / LangChain

| Pattern | How it Works | Pros | Cons / Notes |
|---------|--------------|------|--------------|
| **Plain MemorySaver** | Stores all messages and passes them to the LLM every time. | Simple to implement, keeps full context. | Can grow very large for long conversations, high token usage. |
| **Memory + Trimming (`trim_messages`)** | Keeps only the last N messages or tokens before calling the LLM. | Reduces token usage, keeps recent context. | Older context is lost, may affect reasoning if needed context is trimmed. |
| **Memory + Summarization (`RemoveMessage` + summary)** | When chat history exceeds a threshold, summarizes older messages into a single summary, deletes old messages, and calls LLM with summary + latest message. | Keeps conversation compact, preserves essential context, scales for long sessions. | Slight overhead for summarization calls; summary may lose fine details. |

#### Key Notes

- **Trimming** is useful for short-lived chats or token-limited LLMs.  
- **Summarization** is better for long-term memory in multi-turn conversations.  
- You can combine both strategies: summarize old messages and trim the very latest ones if token usage is still high.  
- `MemorySaver` acts as a simple checkpointer to store/retrieve message history for your workflow.